# model.ipynb — trening klasyfikatora skali downscalingu

Kontynuacja `pipeline.ipynb`. Pipeline generuje dataset i eksportuje `1K_out.zip` na Google Drive — ten notebook pobiera ten ZIP, trenuje model CNN i odkłada wyniki do tego samego folderu na Drive.

**Przepływ:**
1. Montowanie Drive (te same `FOLDER_ID` / `FOLDER_NAME` co w pipeline)
2. Konfiguracja — jedyne miejsce do edycji, te same nazwy co w KROK 3 pipeline'u
3. Rozpakowanie `1K_out.zip` z Drive na lokalny dysk Colaba
4. Trening `cnn_downscale_classifier` z tuningiem Optuna
5. Zapis modelu + artefaktów z powrotem do folderu `1K_out` na Drive

**Architektura CNN:** 6 bloków `Conv→BN→MaxPool` (Frackiewicz et al., Sensors 2025), lokalna normalizacja MSCN, głowica softmax na 4 klasy (CTRL / 8× / 16× / 32×).

## 1. Montowanie Folderu


In [ ]:
%pip install gdown

# --- DLA CAŁEGO FOLDERU ---
FOLDER_LINK = "1Y2xs6l4WMUqnbiGJLZRur4upDjcSpgGX"
!gdown --folder https://drive.google.com/drive/u/1/folders/1Y2xs6l4WMUqnbiGJLZRur4upDjcSpgGX

# --- DLA POJEDYNCZEGO PLIKU (np. zip) ---
# Skopiuj z linku samo ID pliku (ciąg znaków pomiędzy /d/ a /view)
# !gdown --id TUTAJ_WKLEJ_ID_PLIKU

## 2. Konfiguracja — **jedyne miejsce do edycji**

Wartości muszą zgadzać się z `OUTPUT_DIR` i `DRIVE_DESTINATION` z KROK 3 pipeline'u.  
Po zmianie `OUTPUT_DIR_NAME` w pipeline zmień też tu.  
Ścieżki na Drive są wyprowadzane automatycznie z `path` ustawionego wyżej.

In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil

# === KONFIGURACJA LOKALNA (Serwer komp4) ===
# Automatycznie pobiera ścieżkę, w której aktualnie jesteś (np. /home/ailab/blindIQA_wojnar)
BASE_DIR        = Path(os.getcwd())      

OUTPUT_DIR_NAME = '1K_out'               # Nazwa folderu i ZIPa z danymi
LOCAL_DATA      = BASE_DIR / 'data'      # Gdzie rozpakować dane do treningu (zamiast /content/data)
MODEL_NAME      = 'cnn_downscale_4class.keras'
# === /KONFIGURACJA ===

# Ścieżki wyprowadzone automatycznie
# Zakładamy strukturę: BASE_DIR / 1K_out / 1K_out.zip
DATASET_DIR   = BASE_DIR / OUTPUT_DIR_NAME 
ZIP_PATH      = DATASET_DIR / f'{OUTPUT_DIR_NAME}.zip'

# (Opcjonalnie) Jeśli Twój ZIP leży bezpośrednio w głównym folderze, odkomentuj poniższą linię:
# ZIP_PATH      = BASE_DIR / f'{OUTPUT_DIR_NAME}.zip'

MODEL_OUTPATH = DATASET_DIR / MODEL_NAME

print(f'DATASET_DIR   = {DATASET_DIR}')
print(f'ZIP_PATH      = {ZIP_PATH}')
print(f'LOCAL_DATA    = {LOCAL_DATA}')
print(f'MODEL_OUTPATH = {MODEL_OUTPATH}\n')

# 1. Sprawdzenie, czy plik istnieje na lokalnym dysku
assert ZIP_PATH.exists(), (
    f'❌ Nie znaleziono {ZIP_PATH}\n'
    f'Upewnij się, że plik ZIP został pobrany na serwer i znajduje się we właściwym folderze.'
)
print(f'✅ Archiwum znalezione: {ZIP_PATH.stat().st_size / 1e6:.1f} MB')

# 2. Automatyczne rozpakowanie danych na serwerze (jeśli jeszcze nie rozpakowano)
if not LOCAL_DATA.exists() or not any(LOCAL_DATA.iterdir()):
    print(f"📦 Rozpakowuję archiwum do {LOCAL_DATA}...")
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA)
    print("✅ Rozpakowano pomyślnie.")
else:
    print(f"✅ Dane są już gotowe i rozpakowane w {LOCAL_DATA}.")

## 3. Rozpakowanie datasetu

Rozpakowanie z Drive na lokalny dysk Colaba (znacząco szybsze I/O niż czytanie bezpośrednio z Drive). Idempotentne — jeśli już rozpakowane, pominie.

In [ ]:
import os
import zipfile
from pathlib import Path

# 1. Definicja ścieżek (bazujemy na tym, co ustaliliśmy wcześniej)
# LOCAL_DATA to folder 'data' w Twoim projekcie
MANIFEST_PATH = LOCAL_DATA / 'manifest.jsonl'
DATA_ROOT     = LOCAL_DATA

# 2. Prosta logika: Jeśli nie ma manifestu, rozpakuj ZIPa
if not MANIFEST_PATH.exists():
    print(f"📦 Rozpakowuję {ZIP_PATH.name}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(LOCAL_DATA)
    print("✅ Rozpakowano.")
else:
    print("✅ Dane są już na miejscu.")

# 3. Finalne potwierdzenie
print(f"DATA_ROOT     = {DATA_ROOT}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")

# Szybki test podglądu (opcjonalnie)
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, 'r') as f:
        print(f"Pierwsza linia manifestu: {f.readline().strip()[:100]}...")

## 4. Instalacja zależności i importy

In [ ]:
import os
import sys
import json
import pickle
import warnings
import subprocess
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

import optuna
from optuna.samplers import TPESampler
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import uniform_filter
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# 3. Konfiguracja GPU (Kluczowe na serwerze komp4!)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Pozwala TensorFlow brać tylko tyle pamięci ile potrzebuje, a nie 100% od razu
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        gpu_status = f"Znaleziono {len(gpus)} GPU - Dynamiczne przydzielanie pamięci włączone"
    except RuntimeError as e:
        gpu_status = f"Błąd GPU: {e}"
else:
    gpu_status = "Brak GPU - Obliczenia na CPU"

# 4. Ustawienia wyświetlania i logowania
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'--- STATUS ŚRODOWISKA ---')
print(f'TensorFlow: {tf.__version__}')
print(f'Optuna:     {optuna.__version__}')
print(f'Urządzenie: {gpu_status}')
print(f'Projekt:    {os.getcwd()}')
print(f'-------------------------')

## 5. Hiperparametry treningu

In [ ]:
IMG_SIZE    = 256        # zgodne z target_size_px w pipelinie
NORM_KERNEL = 3          # okno do lokalnej normalizacji (równ. 1-3 z papieru)
NORM_C      = 1.0 / 255.0

N_CLASSES   = 10       
CLASS_NAMES = ['CTRL', '2x', '3x', '4x', '5x', '6x', '7x', '8x','9x', '10x']

# Mapowanie scale_factor → indeks klasy (CTRL niezależnie od sf trafia do klasy 0)
SF_TO_CLASS = {
    1: 0,   # CTRL
    2: 1,   # 2x
    3: 2,   # 3x
    4: 3,   # 4x
    5: 4,   # 5x
    6: 5,   # 6x
    7: 6,   # 7x
    8: 7,   # 8x
    9: 8,   # 9x
    10: 9   # 10x
}

# Trening
N_OPTUNA_TRIALS = 10
EPOCHS_TUNING   = 25
EPOCHS_FINAL    = 80
PATIENCE        = 8
BATCH_SIZE      = 16

# Zakresy hiperparametrów (Tabela 2 z papieru)
HP_SPACE = {
    'n_neurons':    (500, 1000),
    'eta':          (1e-5, 1e-3),
    'dropout_rate': (0.0,  0.8),
}

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUT_DIR = Path('out');  OUT_DIR.mkdir(exist_ok=True)

## 6. Lokalna normalizacja (równ. 1–3 z papieru)

$$\hat I(x,y) = \frac{I(x,y) - \mu(x,y)}{\sigma(x,y) + C}$$

z $\mu, \sigma$ w oknie 3×3, $\omega(i,j) = 1/9$. Wariancję liczymy z tożsamości $\sigma^2 = E[I^2] - \mu^2$.

In [ ]:
def local_normalize(img: np.ndarray, kernel: int = NORM_KERNEL, c: float = NORM_C) -> np.ndarray:
    img = img.astype(np.float32, copy=False)
    out = np.empty_like(img)
    for ch in range(img.shape[2]):
        I   = img[..., ch]
        mu  = uniform_filter(I,     size=kernel, mode='reflect')
        mu2 = uniform_filter(I * I, size=kernel, mode='reflect')
        var = np.maximum(mu2 - mu * mu, 0.0)
        out[..., ch] = (I - mu) / (np.sqrt(var) + c)
    return out


def load_image(image_path) -> np.ndarray:
    """Obraz PNG → (256, 256, 3) po lokalnej normalizacji."""
    img = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.float32) / 255.0
    if img.shape[:2] != (IMG_SIZE, IMG_SIZE):
        img = np.asarray(
            Image.fromarray((img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS),
            dtype=np.float32,
        ) / 255.0
    return local_normalize(img)


# Sanity check
_dummy = np.random.rand(IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
_norm  = local_normalize(_dummy)
print(f'Lokalna normalizacja: shape={_norm.shape}, mean={_norm.mean():+.4f}, std={_norm.std():.4f}')

## 7. Wczytanie manifestu + przypisanie klasy

Klasa wychodzi prosto z parametrów degradacji:
- `downscale.abbr == 'CTRL'` → klasa 0 (niezależnie od `scale_factor` w metadanych)
- inaczej → klasa zależna od `scale_factor` (8 → 1, 16 → 2, 32 → 3)

In [ ]:
def class_from_entry(entry: dict) -> int:
    """Zwraca indeks klasy (0..3) na podstawie pól manifestu."""
    deg = entry['degradation']
    if deg['downscale']['abbr'] == 'CTRL':
        return 0
    return SF_TO_CLASS[deg['scale_factor']]


def load_manifest(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            e = json.loads(line)
            if e.get('is_stress_test'):
                continue
            deg = e['degradation']
            rows.append({
                'sample_id':    e['sample_id'],
                'source_id':    e['source_image_id'],
                'path':         DATA_ROOT / e['file']['path_relative'],
                'split':        e['split'],
                'y_class':      class_from_entry(e),
                'pair_abbr':    f"{deg['downscale']['abbr']}→{deg['upscale']['abbr']}",
                'scale_factor': deg['scale_factor'],
            })
    return pd.DataFrame(rows)


df = load_manifest(MANIFEST_PATH)
print(f'Załadowano {len(df)} próbek\n')
print('Splity:')
print(df['split'].value_counts().to_string(), '\n')

print('Rozkład klas per split (klasa = co model ma przewidzieć):')
ct = pd.crosstab(df['split'], df['y_class'].map(dict(enumerate(CLASS_NAMES))))
ct = ct.reindex(columns=CLASS_NAMES, fill_value=0)
print(ct.to_string(), '\n')

print('Klasa per pair_abbr (kontrola sanity):')
print(pd.crosstab(df['pair_abbr'], df['scale_factor']).to_string())

## 8. tf.data pipeline

Zamiast ładować cały dataset do RAM jako numpy (~11 GB dla 1000 zdjęć), budujemy `tf.data.Dataset` który wczytuje obrazy z dysku lazily — w pamięci jest tylko aktualny batch.

In [ ]:
def load_image_tf(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='lanczos3')
    img = tf.cast(img, tf.float32) / 255.0
    img_b = img[None, ...]
    mu    = tf.nn.avg_pool2d(img_b, NORM_KERNEL, 1, 'SAME')
    mu2   = tf.nn.avg_pool2d(img_b * img_b, NORM_KERNEL, 1, 'SAME')
    sigma = tf.sqrt(tf.maximum(mu2 - mu * mu, 0.0))
    img   = ((img_b - mu) / (sigma + NORM_C))[0]
    return img, label


def make_dataset(df_subset: pd.DataFrame, shuffle: bool = False) -> tf.data.Dataset:
    paths  = df_subset['path'].astype(str).values
    labels = df_subset['y_class'].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(df_subset), seed=SEED, reshuffle_each_iteration=True)

    return (ds
            .map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))


df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'  ].reset_index(drop=True)
df_test  = df[df['split'] == 'test' ].reset_index(drop=True)

ds_train = make_dataset(df_train, shuffle=True)
ds_val   = make_dataset(df_val)
ds_test  = make_dataset(df_test)

y_train = df_train['y_class'].values.astype(np.int32)
y_val   = df_val  ['y_class'].values.astype(np.int32)
y_test  = df_test ['y_class'].values.astype(np.int32)

print(f'train: {len(df_train)} próbek, val: {len(df_val)}, test: {len(df_test)}')

## 9. Architektura — 6 bloków `Conv→BN→MaxPool`

Wszystko jak w papierze (Tabela 1), tylko głowica zmieniona z regresji 1-neuron-linear na **softmax 4-neurony**.

| Blok | Conv filters | Po MaxPool 2×2 |
|------|--------------|-----------------|
| 1    | 32           | 128×128         |
| 2    | 64           | 64×64           |
| 3    | 128          | 32×32           |
| 4    | 256          | 16×16           |
| 5    | 256          | 8×8             |
| 6    | 256          | 4×4             |
| Flatten | — | 4096 |
| Dense | n_neurons (Optuna) | — |
| Dropout | rate (Optuna) | — |
| **Dense softmax** | **N_CLASSES = 4** | — |

In [ ]:
def build_model(n_neurons: int = 512,
                dropout_rate: float = 0.5,
                eta: float = 1e-3) -> keras.Model:
    conv_filters = [32, 64, 128, 256, 256, 256]

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = inp
    for i, f in enumerate(conv_filters):
        x = layers.Conv2D(f, kernel_size=3, padding='same',
                          activation='relu', name=f'conv{i+1}')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}')(x)
        x = layers.MaxPooling2D(pool_size=2, strides=2,
                                padding='valid', name=f'pool{i+1}')(x)

    x = layers.Flatten(name='flatten')(x)
    x = layers.Dense(n_neurons, activation='relu', name='dense_hidden')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    out = layers.Dense(N_CLASSES, activation='softmax', name='downscale_class')(x)

    model = keras.Model(inp, out, name='cnn_downscale_classifier')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=eta),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


# Pokaz architektury
_demo = build_model()
_demo.summary()

## 10. Tuning hiperparametrów — Optuna + TPE

Bayesowska optymalizacja z TPE, zakresy z Tabeli 2 z papieru. Każdy trial: krótki trening, score = best `val_accuracy`. Słabe triale ucinane medianowym prunerem.

> Drogi krok. Na T4/L4 z Colab Pro: ~5-10 min per trial. Możesz zmniejszyć `N_OPTUNA_TRIALS` na 5-8 do szybkiego sanity check.

In [ ]:
class OptunaPruningCallback(callbacks.Callback):
    def __init__(self, trial, monitor='val_accuracy'):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        value = (logs or {}).get(self.monitor)
        if value is None:
            return
        self.trial.report(float(value), epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()


def objective(trial: optuna.Trial) -> float:
    n_neurons    = trial.suggest_int  ('n_neurons',    *HP_SPACE['n_neurons'])
    eta          = trial.suggest_float('eta',          *HP_SPACE['eta'],   log=True)
    dropout_rate = trial.suggest_float('dropout_rate', *HP_SPACE['dropout_rate'])

    print(f'\n  [Trial {trial.number+1}/{N_OPTUNA_TRIALS}] '
          f'n_neurons={n_neurons}, eta={eta:.2e}, dropout={dropout_rate:.2f}')

    keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_model(n_neurons=n_neurons, dropout_rate=dropout_rate, eta=eta)

    class EpochLog(callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            if (epoch + 1) % 5 == 0:
                acc  = (logs or {}).get('accuracy', 0)
                vacc = (logs or {}).get('val_accuracy', 0)
                print(f'    epoch {epoch+1:3d} — acc={acc:.4f}  val_acc={vacc:.4f}')

    hist = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS_TUNING,
        verbose=0,
        callbacks=[
            callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                    patience=PATIENCE, restore_best_weights=True),
            OptunaPruningCallback(trial, monitor='val_accuracy'),
            EpochLog(),
        ],
    )
    best_val = float(max(hist.history['val_accuracy']))
    print(f'  => val_accuracy={best_val:.4f} '
          f'(epok: {len(hist.history["val_accuracy"])})')
    return best_val


def _log_trial(study, trial):
    if trial.state == optuna.trial.TrialState.PRUNED:
        print(f'  => PRUNED')
    if study.best_trial.number == trial.number:
        print(f'  ** Nowy najlepszy trial! **')


study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f'Start tuningu: {N_OPTUNA_TRIALS} triali x maks. {EPOCHS_TUNING} epok')
print('=' * 60)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[_log_trial])

print('\n' + '=' * 60)
print('=== Najlepszy trial ===')
print(f'  val_accuracy  = {study.best_value:.4f}')
print(f'  n_neurons     = {study.best_params["n_neurons"]}')
print(f'  eta           = {study.best_params["eta"]:.5f}')
print(f'  dropout_rate  = {study.best_params["dropout_rate"]:.3f}')

with open(OUT_DIR / 'best_params.json', 'w') as f:
    json.dump(study.best_params, f, indent=2)

## 11. Finalny trening

In [ ]:
keras.backend.clear_session()
tf.random.set_seed(SEED)

best = study.best_params
model = build_model(
    n_neurons    = best['n_neurons'],
    dropout_rate = best['dropout_rate'],
    eta          = best['eta'],
)

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS_FINAL,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                patience=PATIENCE, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_accuracy', mode='max',
                                    factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ],
)
print('\nTrening zakończony')

## 12. Ewaluacja — accuracy + confusion matrix + per-class

In [ ]:
def evaluate(model, ds, y_true, name='test') -> dict:
    proba  = model.predict(ds, verbose=0)
    y_pred = proba.argmax(axis=1)

    acc = accuracy_score(y_true, y_pred)
    print(f'=== {name.upper()} — accuracy = {acc:.4f} ===\n')
    print('Per-class metrics:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    return {'y_pred': y_pred, 'proba': proba, 'accuracy': acc}


res_val  = evaluate(model, ds_val,  y_val,  'val')

res_test = evaluate(model, ds_test, y_test, 'test')

### 12a. Confusion matrix + dokładność per algorytm

Sprawdzamy, czy model nie myli się systematycznie na konkretnej parze downscale→upscale (np. czy ESRGAN rzeczywiście "ukrywa" downscaling lepiej niż BIC).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix (test)
cm = confusion_matrix(y_test, res_test['y_pred'], normalize='true')
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap='Blues', values_format='.2f', colorbar=False
)
axes[0].set_title(f"Confusion matrix — test (acc = {res_test['accuracy']:.3f})")

# Krzywe uczenia
axes[1].plot(history.history['accuracy'],     label='train', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='val',   linewidth=2)
axes[1].set_xlabel('Epoka'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Krzywe uczenia')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'eval.png', dpi=120, bbox_inches='tight')
plt.show()

# Accuracy per (pair_abbr × scale_factor)
df_eval = df_test.copy()
df_eval['y_pred'] = res_test['y_pred']
df_eval['correct'] = (df_eval['y_pred'] == df_eval['y_class']).astype(int)

print('\nAccuracy per (pair_abbr × scale_factor):')
acc_table = (df_eval.groupby(['pair_abbr', 'scale_factor'])['correct']
                   .agg(['count', 'mean'])
                   .rename(columns={'mean': 'accuracy'})
                   .round(3))
print(acc_table.to_string())

## 13. Zapis modelu na Drive

Model + metadane lądują w tym samym folderze co `.zip` (`DRIVE_BASE`).

In [ ]:
# Lokalnie najpierw — szybki zapis
local_model = OUT_DIR / MODEL_NAME
model.save(local_model)

# Metadane
artifacts = {
    'best_params':   study.best_params,
    'val_accuracy':  float(res_val['accuracy']),
    'test_accuracy': float(res_test['accuracy']),
    'class_names':   CLASS_NAMES,
    'sf_to_class':   SF_TO_CLASS,
    'history':       {k: [float(v) for v in vals] for k, vals in history.history.items()},
}
with open(OUT_DIR / 'artifacts.json', 'w') as f:
    json.dump(artifacts, f, indent=2)

# Kopia na Drive — ten sam folder co ZIP z pipeline
if ON_COLAB:
    DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_model, MODEL_OUTPATH)
    shutil.copy2(OUT_DIR / 'artifacts.json', DRIVE_DATASET_DIR / 'artifacts.json')
    print(f'Model skopiowany na Drive: {MODEL_OUTPATH}')
    print(f'Artefakty: {DRIVE_DATASET_DIR / "artifacts.json"}')
else:
    print(f'Model zapisany lokalnie: {local_model}')

## 14. Inferencja na nowym obrazie

In [ ]:
def predict_downscale(image_path, model=model) -> dict:
    """Zwraca przewidywaną klasę + prawdopodobieństwa per klasa."""
    x = load_image(image_path)[None, ...]
    proba = model.predict(x, verbose=0)[0]
    cls = int(proba.argmax())
    return {
        'predicted_class':      CLASS_NAMES[cls],
        'predicted_class_idx':  cls,
        'probabilities':        dict(zip(CLASS_NAMES, [float(p) for p in proba])),
    }


# Demo
demo_row = df_test.iloc[0]
result = predict_downscale(demo_row['path'])
print(f'Obraz:        {demo_row["sample_id"]}')
print(f'Degradacja:   {demo_row["pair_abbr"]}  sf={demo_row["scale_factor"]}')
print(f'Prawdziwa kl: {CLASS_NAMES[demo_row["y_class"]]}')
print(f'Predykcja:    {result["predicted_class"]}')
print(f'P(klas):      ' + ', '.join(f'{k}={v:.3f}' for k, v in result['probabilities'].items()))